# Export and the 3D globe

This notebook exports a world for use outside worldgen: global rasters of any map field, zoomed regions with their culture and engine heightmaps, a tile pyramid from the globe down to local maps, meshes and a 3D globe. It also shows the same exports from the command line.

Run it from the `notebooks/` folder after `pip install -e ".[dev,notebooks,export,globe]"`. GeoTIFF needs the `export` extra (rasterio); PNG and RAW work without it.

In [ ]:
%matplotlib inline
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from worldgen import generate_world, load_spec
from worldgen.export import EXPORT_FIELDS, export_global, global_colours, global_raster
from worldgen.export.writers import decode, read_sidecar

out = Path(tempfile.mkdtemp(prefix="worldgen_export_"))

## 1. A planet

The Earth example on the standard grid (40k cells, ~124 km) with the full plate simulation (~30 s).

In [ ]:
world = generate_world(load_spec("../examples/earth.yaml"), resolution="standard")
print(world.state.name, f"radius {world.surface.attrs['radius_m'] / 1e3:.0f} km")

## 2. Global rasters

`export_global` samples a field onto a latitude–longitude raster `width` × `width/2`, north up and starting at 180° W. Continuous fields are blended linearly within the grid's triangles, so the raster reproduces the grid values at the cells; categories take the nearest cell. The format follows the file suffix:

- `.tif`: GeoTIFF, 32-bit floats (or integer codes), georeferenced on a sphere of the planet's radius;
- `.png`: 16-bit codes spanning the field's range (the world's whole elevation range for heights);
- `.r16` or `.raw`: the same 16-bit codes, little-endian, for terrain editors.

Each file gets a JSON sidecar (`<file>.json`) with the units, the code-to-value mapping, category labels and the georeferencing.

In [ ]:
for name in ("elevation.tif", "elevation.png", "elevation.r16"):
    path = export_global(world, out / name, "elevation", width=2048)
    print(f"{name:15s} {path.stat().st_size / 1e6:5.1f} MB  {read_sidecar(path)['encoding']}")

The PNG codes decode back to metres with the sidecar's offset and scale:

In [ ]:
from PIL import Image

meta = read_sidecar(out / "elevation.png")
heights = decode(np.asarray(Image.open(out / "elevation.png")), meta["encoding"])
values = global_raster(world, "elevation", width=2048).data
print(f"largest decoding error {np.abs(heights - values).max():.2f} m (step {meta['encoding']['scale']:.2f} m)")
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(heights, extent=(-180, 180, -90, 90), cmap="terrain", vmin=-6000, vmax=6000)
fig.colorbar(im, ax=ax, label="elevation (m)", shrink=0.7)
ax.set_xlabel("longitude (°)")
ax.set_ylabel("latitude (°)");

## 3. Categories and other fields

Categories are written as integer codes with their labels in the sidecar. Every map field can be exported, plus `surface` (ice included), `ocean`, `lake`, `rivers`, `discharge`, `runoff`, `sea_ice`, `vegetation` and `sediment`; monthly climate fields are `temperature:<month>` and `rainfall:<month>`. Culture fields (with `epoch=` for a kept epoch of an organic history) need a world with a culture.

In [ ]:
print(", ".join(EXPORT_FIELDS))
path = export_global(world, out / "biomes.png", "biomes", width=2048)
labels = read_sidecar(path)["labels"]
codes = np.asarray(Image.open(path))
area = np.cos(np.radians(np.linspace(90, -90, codes.shape[0] + 1)[:-1] - 90 / codes.shape[0]))[:, None] * np.ones_like(codes)
for code in np.unique(codes):
    print(f"{labels[str(code)]:28s} {area[codes == code].sum() / area.sum():6.1%} of the surface")

## 4. Map colours

With `colour=True` the raster holds the colours the field's map uses (RGB, optionally hillshaded), for use as a texture.

In [ ]:
path = export_global(world, out / "elevation_colour.png", "elevation", width=2048, colour=True, hillshade=True)
texture = global_colours(world, "biomes", width=2048).data
fig, axes = plt.subplots(2, 1, figsize=(10, 10))
axes[0].imshow(Image.open(path), extent=(-180, 180, -90, 90))
axes[1].imshow(texture, extent=(-180, 180, -90, 90))
for ax in axes:
    ax.set_xlabel("longitude (°)")
    ax.set_ylabel("latitude (°)")
print("files in", out)

## 5. A zoomed region

Regions are exported on their own cube-face lattice, without resampling, so neighbouring regions meet exactly. A file holds the lattice with its high-t edge on top and is georeferenced by ground control points through its nodes; GIS tools should warp it with a thin-plate spline (`gdalwarp -tps`). `crs="laea"` writes a north-up copy resampled onto a Lambert azimuthal equal-area grid centred on the region. Heights in PNG and RAW share the world's height scale (its elevation range widened by 15% each side), so global and regional files line up.

`export_region_bundle` writes a folder: the requested fields, the local map's colours, every node's longitude and latitude, and, given the region's culture, polity, culture, settlement-footprint, road and habitat rasters plus GeoJSON settlements, roads and borders.

In [ ]:
from worldgen.culture import CultureSpec, apply_culture
from worldgen.culture.local import zoom_culture
from worldgen.export import export_region_bundle
from worldgen.zoom import zoom_region

world = apply_culture(world, CultureSpec.model_validate({"cultures": [{"name": "Ana", "development": {"energy": "electric"}}]}))
towns = world.culture.settlements
k = int(np.argsort(towns.population)[-200])
region = zoom_region(world, towns.lat_deg[k], towns.lon_deg[k])
local = zoom_culture(world, region)
folder = export_region_bundle(world, region, out / "region", culture=local)
print(f"{region.grid.shape[0]}² nodes, {len(local)} settlements, {len(local.roads)} road segments")
print(json.dumps(json.loads((folder / "region.json").read_text())["files"], indent=1))

The colours with the GeoJSON borders and largest settlements placed back on the image through the file's control points:

In [ ]:
import rasterio
from rasterio.transform import GCPTransformer

with rasterio.open(folder / "colour.tif") as src:
    image = np.moveaxis(src.read(), 0, -1)
    to_image = GCPTransformer(src.gcps[0], tps=True)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(image)
for feature in json.loads((folder / "borders.geojson").read_text())["features"]:
    lon, lat = np.array(feature["geometry"]["coordinates"]).T
    r, c = to_image.rowcol(list(lon), list(lat), op=lambda v: v)
    ax.plot(np.asarray(c) - 0.5, np.asarray(r) - 0.5, color="0.15", lw=0.7)
places = json.loads((folder / "settlements.geojson").read_text())["features"]
for feature in sorted(places, key=lambda f: -f["properties"]["population"])[:10]:
    lon, lat = feature["geometry"]["coordinates"]
    r, c = to_image.rowcol([lon], [lat], op=lambda v: v)
    ax.plot(c[0] - 0.5, r[0] - 0.5, "o", color="crimson", ms=3)
    ax.annotate(feature["properties"]["name"], (c[0], r[0]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_axis_off()

## 6. Engine heightmaps

Unity terrains take 2ⁿ + 1 samples per side, Unreal landscapes 127, 253, 505, 1009, 2017, 4033 or 8129. `engine_region` builds the region on a lattice of 129 (Unity) or 127 (Unreal) nodes per tile, so the sizes come out exactly without resampling; `export_engine_heightmap` writes 16-bit RAW or PNG and puts the import settings (size, metres per sample, height range or Z scale) in the sidecar. Blocks of up to 8 × 8 tiles fit the zoom's node limit (1025 for Unity, 1009 for Unreal).

In [ ]:
from worldgen.export import engine_region, engine_sizes, export_engine_heightmap

print("Unity sizes", engine_sizes("unity"), "Unreal sizes", engine_sizes("unreal"))
terrain = engine_region(world, towns.lat_deg[k], towns.lon_deg[k], "unity", 513)
path = export_engine_heightmap(world, terrain, out / "terrain.r16", "unity")
print(json.dumps(read_sidecar(path)["engine_import"], indent=1))

## 7. A tile pyramid

The pyramid holds a world's heights at every quadtree level, in tiles of 129 × 129 nodes. Its global part is built at a top level for all six cube faces (zoom detail and erosion), re-levelled onto the global surface (zoom erosion lowers coarse lattices more than fine ones), stitched where faces meet, and the coarser levels keep every other node. Chains add finer levels around a place: each level is the 2 × 2 tiles over its parent tile, re-levelled onto the parent and ramped to the parent's heights over a band at its edge, so it meets the coarser tiles around it without cracks. The default top level is 3 (~9 km, ~6.5 min for the globe); this notebook uses level 2 (~18 km) to stay quick.

In [ ]:
from worldgen.export import add_chain, build_global, export_pyramid

pyramid = build_global(world, top_level=2)
add_chain(pyramid, world, towns.lat_deg[k], towns.lon_deg[k], 7)
folder = export_pyramid(pyramid, world, out / "pyramid")
index = json.loads((folder / "tileset.json").read_text())
for level, entry in index["levels"].items():
    print(f"level {level}: {len(entry['tiles']):4d} tiles, nominal step {entry['nominal_step_m'] / 1e3:6.2f} km")

The chain's levels side by side, each the 2 × 2 tiles over the one before (the band where each level meets its parent is visible as smoother ground along its edge):

In [ ]:
from matplotlib.colors import LightSource

levels = range(3, 8)
fig, axes = plt.subplots(1, len(levels), figsize=(3 * len(levels), 3))
for ax, level in zip(axes, levels):
    (block,) = [b for b in pyramid.blocks(level) if b.coverage == "chain"]
    z = block.fields["elevation"]
    ax.imshow(LightSource(315, 40).hillshade(z, vert_exag=0.05, dx=1, dy=1), cmap="gray", origin="lower")
    ax.set_axis_off()
    ax.text(0.03, 0.03, f"level {level}", transform=ax.transAxes, color="yellow", fontsize=8)

## 8. Meshes

Meshes are built per tile and simplified with a right-triangulated irregular network whose border nodes are always kept, so neighbouring tiles share their border vertices and meet without cracks. The error the simplification allows (default 5% of the node spacing) includes how far each chord sags below the sphere. The sea floor stays in the terrain mesh and a separate sea-surface mesh repeats the terrain triangles that touch the sea, so the coast is where the two cross. Vertices are in east–north–up metres around each tile's centre; glTF files place each tile by a node transform, OBJ and PLY files take planet-centred or local metres. Without a pyramid, `globe_meshes` resamples the global grid onto the cube-sphere (a few seconds).

In [ ]:
from worldgen.export import export_globe_mesh, export_region_mesh, globe_meshes, pyramid_meshes, write_meshes

quick = export_globe_mesh(world, out / "globe_quick.glb", level=2)                  # from the global grid
detailed = write_meshes(pyramid_meshes(pyramid, world, 2, field="biomes", exaggeration=20), out / "globe_biomes.glb")
local = export_region_mesh(world, terrain, out / "terrain.obj", max_error_m=20)       # the Unity block of section 6
for path in (quick, detailed, local):
    print(f"{path.name:18s} {path.stat().st_size / 1e6:6.1f} MB")

The Unity block as a mesh simplified to 20 m, drawn with its height above the sphere (×3) against its local east and north coordinates:

In [ ]:
from worldgen.export import region_meshes

(mesh, *_) = region_meshes(world, terrain, max_error_m=20, sea=False)
v = mesh.vertices
height = np.linalg.norm(mesh.planet_vertices(), axis=1) - terrain.radius_m
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_trisurf(v[:, 0] / 1e3, v[:, 1] / 1e3, height * 3 / 1e3, triangles=mesh.faces, color="0.7", linewidth=0.05,
                edgecolor="0.3", shade=True)
ax.set_xlabel("east (km)")
ax.set_ylabel("north (km)")
ax.set_zlabel("height ×3 (km)")
print(f"{len(mesh.faces)} triangles from {terrain.grid.size} nodes")

## 9. The 3D globe

`GlobeViewer` builds the globe once (heights ×20 by default) and recolours it for any map field and any kept epoch of an organic history; zoomed regions are draped over it, raised a few metres. `show()` opens a window (keys n and p step through the fields, a slider picks the epoch), `widget()` gives field and epoch menus over the interactive globe in Jupyter (PyVista's trame backend), `screenshot()` writes an image and `export_views()` standalone HTML pages or images. Here the world gets an organic culture with a 3000-year history so there are epochs to show.

In [ ]:
import pyvista as pv
from IPython.display import Image as Picture
from worldgen.render import GlobeViewer

pv.OFF_SCREEN = True          # images here; leave this out to open windows or use the widget
history = {"history": {"start_years_ago": 3000, "epochs": [1500]},
           "cultures": [{"name": "Oru", "mode": "organic", "homeland": {"lat_deg": float(towns.lat_deg[k]),
                                                                          "lon_deg": float(towns.lon_deg[k]),
                                                                          "radius_km": 3000}}]}
old_world = apply_culture(world, CultureSpec.model_validate(history))
globe = GlobeViewer(old_world, level=2)
globe.add_region(region)
print("epochs", globe.dates)
print("fields", ", ".join(globe.fields))

In [ ]:
shots = []
for field, years in (("elevation", 0), ("biomes", 0), ("energy", 1500), ("energy", 0)):
    shots.append(globe.screenshot(out / f"globe_{field}_{years}.png", field, years, lat_deg=float(towns.lat_deg[k]),
                                  lon_deg=float(towns.lon_deg[k]), distance=2.4, window_size=(500, 500)))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, path in zip(axes, shots):
    ax.imshow(plt.imread(path))
    ax.set_axis_off()

Interactive use (needs a display; the widget needs a Jupyter server with PyVista's trame backend):

In [ ]:
pv.OFF_SCREEN = False
globe.widget()                      # in Jupyter, with PyVista's trame backend
globe.show("political", 1500)       # a window from a script
globe.save_html(out / "political.html", "political", 1500)  # one interactive page
globe.export_views(out / "views", ("political", "energy"))   # an HTML page per field and kept epoch

## 10. The command line

`worldgen export` has a subcommand per kind of export (`raster`, `region`, `engine`, `pyramid`, `mesh`, and `fields` to list what a world can export); `worldgen globe` opens the globe in a window or writes an image (`.png`), an interactive page (`.html`) or a folder of views. They work on a saved world:

In [ ]:
from worldgen import save_world

folder = save_world(old_world, out / "world")
place = f"--lat={float(towns.lat_deg[k]):.3f} --lon={float(towns.lon_deg[k]):.3f}"
!worldgen export fields {folder} | head -5
!worldgen export raster {folder} {out}/cli_rainfall.tif --field rainfall --width 1024
!worldgen export engine {folder} {out}/cli_unity.r16 {place} --size 257
!worldgen globe {folder} {out}/cli_views --field elevation,political --all-epochs --level 1 2>/dev/null

Other examples (not run here; the pyramid takes ~7 min at the default top level 3):

```
worldgen export region world/ alps/ --lat 46.5 --lon 9 --format .png --crs laea
worldgen export pyramid world/ tiles/ --place 46.5,9,7 --fields elevation,rivers --meshes
worldgen export mesh world/ globe.glb --level 2 --field biomes --exaggeration 20
worldgen export mesh world/ alps.glb --lat 46.5 --lon 9 --exaggeration 2
worldgen globe world/ globe.html --field political --epoch 1500
worldgen globe world/                  # a window
```

`scripts/validate_export.py` checks every export of a world (file round trips, georeferencing against GDAL's thin-plate warp, seams between faces, levels and mesh tiles, the simplification tolerance) and reports timings; `--pyramid` adds the full pyramid.